<a href="https://colab.research.google.com/github/Alilson2/Projeto_IA/blob/main/GerarDataFrame.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr # Ler arquivos netcdf
from google.colab import drive
import seaborn as sns


from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import matplotlib.patches as mpatches # Desenhar geometria em um mapa

In [2]:
import os
import xarray as xr

# --- Clonar repositório se não existir ---
if not os.path.exists("Projeto_IA"):
    !git clone https://github.com/Alilson2/Projeto_IA.git
else:
    print("📁 Repositório 'Projeto_IA' já existe — pulando o clone.")

# --- Verificar se a pasta foi criada ---
if not os.path.exists("Projeto_IA"):
    raise FileNotFoundError("❌ A pasta 'Projeto_IA' não foi encontrada. O clone pode ter falhado.")
else:
    print("\n✅ Repositório clonado com sucesso!\n")
    print("Arquivos dentro da pasta Projeto_IA:\n", os.listdir("Projeto_IA"))

# --- Localizar arquivos .nc ---
arquivos_nc = [f for f in os.listdir("Projeto_IA") if f.endswith(".nc")]
if not arquivos_nc:
    raise FileNotFoundError("❌ Nenhum arquivo .nc encontrado na pasta Projeto_IA!")
else:
    print("\n📂 Arquivo(s) NetCDF encontrado(s):")
    #for f in arquivos_nc:
        #print(" -", f)

# --- Montar lista de caminhos ---
ARQUIVO = [os.path.join("Projeto_IA", f) for f in arquivos_nc]

# --- Função para corrigir longitude ---
def corrigir_longitude(ds):
    for coord in ["longitude", "lon"]:
        if coord in ds.coords:
            ds = ds.assign_coords({coord: ((ds[coord] + 180) % 360) - 180})
            ds = ds.sortby(coord)
    return ds

# --- Abrir arquivos com segurança (nova sintaxe) ---
try:
    dados = xr.open_mfdataset(
        ARQUIVO,
        combine='by_coords',
        parallel=True,           # usa múltiplos núcleos
        preprocess=corrigir_longitude,
        combine_attrs='override' # 🟢 substitui o antigo compat='override'
    )
except ValueError as e:
    print("\n⚠️ Erro na combinação — tentando modo 'nested' (concat por tempo)...")
    dados = xr.open_mfdataset(
        ARQUIVO,
        combine='nested',
        concat_dim='valid_time',  # ajuste se sua dimensão temporal tiver outro nome
        parallel=True,
        preprocess=corrigir_longitude,
        combine_attrs='override'
    )

print("\n✅ Dataset carregado com sucesso!\n")

Cloning into 'Projeto_IA'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 269 (delta 49), reused 24 (delta 24), pack-reused 205 (from 1)
Receiving objects: 100% (269/269), 165.85 MiB | 16.59 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Updating files: 100% (129/129), done.

✅ Repositório clonado com sucesso!

Arquivos dentro da pasta Projeto_IA:
 ['(JANEIRO 2023 - 1a15) e65d028998b4350fbae18fbb65938bdb.nc', 'AGO 2022 - 16a31.nc', '(DEZ 2020-25 a 31).nc', 'ProjetoIA_ver2.ipynb', '(ABR 2020-1 a 24).nc', '(DEZEMBRO 2024 - 1a15) eb8accab077c6a6dc49976db8aecbc00.nc', '(MARÇO 2021-16 a 31) .nc', '(FEVEREIRO 2023 - 1a15) 1c71b07bf4a2f87d503280bd7fdd509b.nc', '(JAN 2024 - 16a31) ea884d573d8753c6f2976f9e4b642c11.nc', '(AGOSTO 2021-16 a 31).nc', 'Copy_of_ProjetoIA_ver2.ipynb', '(DEZEMBRO 2023 - 1a15).nc', '(ABRIL 2024 - 1a15) cbec3732f428a3c38f998555ad01121d.nc', '(AGOSTO 2024 - 1a15) 1d

/tmp/ipython-input-3327130302.py:39: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  dados = xr.open_mfdataset(
/tmp/ipython-input-3327130302.py:39: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'latitude' ('latitude',) The recommendation is to set join explicitly for this case.
  dados = xr.open_mfdataset(



⚠️ Erro na combinação — tentando modo 'nested' (concat por tempo)...


/tmp/ipython-input-3327130302.py:48: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'latitude' ('latitude',) The recommendation is to set join explicitly for this case.
  dados = xr.open_mfdataset(
/tmp/ipython-input-3327130302.py:48: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'longitude' ('longitude',) The recommendation is to set join explicitly for this case.
  dados = xr.open_mfdataset(



✅ Dataset carregado com sucesso!



In [3]:
df = dados.to_dataframe()
df = df.dropna()

# --- 2️⃣ Verificar se há a coordenada temporal ---
if "valid_time" not in dados.coords:
    raise ValueError("❌ O dataset não contém uma coordenada temporal chamada 'valid_time'.")

# --- 3️⃣ Converter o eixo temporal para pandas.DatetimeIndex ---
tempo = pd.to_datetime(dados["valid_time"].values)

# --- 4️⃣ Criar DataFrame com componentes temporais ---
df_tempo = pd.DataFrame({
    "timestamp": tempo,
    "timestamp_segundos": tempo.view("int64"),   # segundos desde 1970
    "ano": tempo.year,
    "mes": tempo.month,
    "dia": tempo.day,
    "hora": tempo.hour,
    "minuto": tempo.minute,
    "segundo": tempo.second,
    "dia_semana": tempo.dayofweek,
    "dia_do_ano": tempo.dayofyear
})

#print(df_tempo.head())

# Exemplo: seleciona uma variável e um período
# Sort the dataset by valid_time before slicing
dados_sorted = dados.sortby('valid_time')
dados_filtrado = dados_sorted

# Converte para pandas sem estourar RAM
df_panda = dados_filtrado.to_dataframe().reset_index()
df_panda = df_panda.dropna()

# Junta com df_tempo
df_final = pd.merge(
    df_panda,
    df_tempo,
    left_on='valid_time',
    right_on='timestamp',
    how='left'
)


df_final['longitude'] = np.round(df_final['longitude'], 1)
df_final['latitude'] = np.round(df_final['latitude'], 1)

In [6]:
def selecionar_coordenadas(df, N_lim, S_lim, L_lim, O_lim):

    df = df[
        (df["latitude"] <= N_lim) &
        (df["latitude"] >= S_lim) &
        (df["longitude"] >= O_lim) &
        (df["longitude"] <= L_lim)
    ]

    df = df.drop_duplicates(subset=["valid_time", "latitude", "longitude"])
    return df


def calcula_vapor_umidade(df):

    t2m_c = df["t2m"] - 273.15
    d2m_c = df["d2m"] - 273.15

    es = 610.94 * np.exp(17.625 * t2m_c / (243.04 + t2m_c))
    ev = 610.94 * np.exp(17.625 * d2m_c / (243.04 + d2m_c))

    df["es"]  = es
    df["ev"]  = ev
    df["RH"]  = 100 * ev / es
    df["VPD"] = (es - ev) / 1000

    return df


def criar_dados(df):

    df["data"] = pd.to_datetime(df["valid_time"]).dt.date

    # Variáveis com estatísticas: mean, min, max
    vars_stats = ['d2m', 't2m', 'u10', 'v10', 'sp', 'es', 'ev', 'RH', 'VPD']

    # Variáveis que usam somente o último valor do dia
    vars_last = ["e", "ssrd", "sshf", "slhf", "tp"]

    g = df.groupby(["data", "latitude", "longitude"])

    resultados = {}

    # ---------- Estatísticas ----------
    for var in vars_stats:
        resultados[f"{var}_mean"] = g[var].mean()
        resultados[f"{var}_min"]  = g[var].min()
        resultados[f"{var}_max"]  = g[var].max()

    # ---------- Último valor ----------
    for var in vars_last:
        resultados[f"{var}_last"] = g[var].last()

    df_pixel = pd.concat(resultados, axis=1).reset_index()
    return df_pixel


def agregar_regional(df_pixel):

    df_pixel["data"] = pd.to_datetime(df_pixel["data"])
    df_pixel["mes"]  = df_pixel["data"].dt.month

    g = df_pixel.groupby("data")
    resultados = {}

    for col in df_pixel.columns:
        if col in ["data", "latitude", "longitude", "mes"]:
            continue

        # Últimos valores → média espacial
        if col.endswith("_last"):
            var = col.replace("_last", "")
            resultados[f"{var}_mean"] = g[col].mean()
            continue

        # Estatísticas (mean/min/max por pixel por dia)
        if "_mean" in col:
            var = col.replace("_mean", "")
            resultados[f"{var}_mean"] = g[col].mean()
            resultados[f"{var}_std"]  = g[col].std()
            continue

        if "_min" in col:
            var = col.replace("_min", "")
            resultados[f"{var}_min"]  = g[col].min()
            continue

        if "_max" in col:
            var = col.replace("_max", "")
            resultados[f"{var}_max"]  = g[col].max()
            continue

    df_regional = pd.concat(resultados, axis=1).reset_index()
    df_regional["mes"] = df_regional["data"].dt.month

    return df_regional


def pipeline(df_raw, N_lim, S_lim, L_lim, O_lim):

    df = selecionar_coordenadas(df_raw, N_lim, S_lim, L_lim, O_lim)
    df = calcula_vapor_umidade(df)
    df_daily_pixel = criar_dados(df)
    df_regional = agregar_regional(df_daily_pixel)

    return df_regional


def remover_correlacoes(X, threshold=0.90):
    corr_matrix = X.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]

    print(f"\n[Remoção por correlação] Features removidas (>{threshold}): {len(to_drop)}")
    print(to_drop)

    return X.drop(columns=to_drop), to_drop


def diagnosticar_overfitting(model, X_train, y_train, X_test, y_test):
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)

    acc_train = accuracy_score(y_train, y_pred_train)
    acc_test  = accuracy_score(y_test, y_pred_test)
    diff = acc_train - acc_test

    cv_scores = cross_val_score(model, X_train, y_train, cv=10)
    cv_mean = cv_scores.mean()

    print("\n================ DIAGNÓSTICO DE OVERFITTING ================")
    print(f"Acurácia Treino: {acc_train:.4f}")
    print(f"Acurácia Teste:  {acc_test:.4f}")
    print(f"Diff Treino - Teste: {diff:.4f}")
    print("-------------------------------------------------------------")
    print(f"Cross-Validation (média 10 folds): {cv_mean:.4f}")
    print(f"Desvio CV: {cv_scores.std():.4f}")
    print("-------------------------------------------------------------")

    if diff < 0.05:
        status = "OK — Sem sinais de overfitting."
    elif diff < 0.10:
        status = "Atenção — Overfitting leve."
    elif diff < 0.20:
        status = "Overfitting moderado."
    else:
        status = "Overfitting severo — modelo está decorando os dados."

    print("Diagnóstico:", status)
    print("=============================================================\n")



def separar_dados(df_diario, limiar = 0.0005):
  X = df_diario.drop(['data', 'tp_mean'], axis=1)
  y = df_diario['tp_mean'].copy()

  ind_sem_chuva = np.where(y <= limiar)[0]
  ind_com_chuva = np.where(y > limiar)[0]

  y.iloc[ind_sem_chuva] = 0
  y.iloc[ind_com_chuva] = 1

  return X, y

def data_treino_teste(df_diario):
  ind_per = int(0.8 * len(df_diario))
  df_train = df_diario.iloc[:ind_per].copy()
  df_test  = df_diario.iloc[ind_per:].copy()

  df_train["tp_mean"] = df_train["tp_mean"].shift(-1)
  df_test["tp_mean"]  = df_test["tp_mean"].shift(-1)

  df_train = df_train.dropna().reset_index(drop=True)
  df_test  = df_test.dropna().reset_index(drop=True)
  return df_train, df_test

def selecionar_features(X_train, X_test, y_train, k=15):
  rf_temp = RandomForestClassifier(
      n_estimators=200,
      max_depth=8,
      random_state=42#,
      #class_weight='balanced_subsample'
  )

  rf_temp.fit(X_train, y_train)

  importances = pd.DataFrame({
      'feature': X_train.columns,
      'importance': rf_temp.feature_importances_
  }).sort_values('importance', ascending=False)

  print("\n[IMPORTÂNCIA DAS FEATURES]")
  print(importances.head(20))

  k = 15
  top_k_features = importances.head(k)["feature"].tolist()

  print(f"\nTop {k} features selecionadas:")
  print(top_k_features)

  X_train_sel = X_train[top_k_features]
  X_test_sel  = X_test[top_k_features]
  return X_train_sel, X_test_sel

In [7]:

O_lim = -46.84
L_lim = -46.36
N_lim = -23.4
S_lim = -23.74

df_panda = df_final
df_diario = pipeline(df_panda, N_lim, S_lim, L_lim, O_lim)

In [9]:
df_train, df_test = data_treino_teste(df_diario)

X_train_raw, y_train = separar_dados(df_train)
X_test_raw,  y_test  = separar_dados(df_test)

X_train_corr, removed_corr = remover_correlacoes(X_train_raw, threshold=0.90)
X_test_corr = X_test_raw.drop(columns=removed_corr)


scaler = StandardScaler()
scaler.fit(X_train_corr)

X_train = pd.DataFrame(scaler.transform(X_train_corr), columns=X_train_corr.columns)
X_test  = pd.DataFrame(scaler.transform(X_test_corr),  columns=X_train_corr.columns)


#X_train, X_test = selecionar_features(X_train, X_test, y_train, k=15)



[Remoção por correlação] Features removidas (>0.9): 19
['d2m_min', 'd2m_max', 't2m_min', 'u10_max', 'v10_min', 'sp_min', 'sp_max', 'es_mean', 'es_std', 'es_min', 'es_max', 'ev_mean', 'ev_std', 'ev_min', 'ev_max', 'VPD_mean', 'VPD_min', 'VPD_max', 'slhf_mean']

[IMPORTÂNCIA DAS FEATURES]
      feature  importance
0    d2m_mean    0.127968
15     RH_min    0.102827
11    sp_mean    0.098219
2    t2m_mean    0.062141
13    RH_mean    0.055821
1     d2m_std    0.054289
10    v10_max    0.052903
4     t2m_max    0.040359
19  ssrd_mean    0.039796
18     e_mean    0.038491
21        mes    0.037207
5    u10_mean    0.033917
8    v10_mean    0.032672
6     u10_std    0.031954
12     sp_std    0.030957
14     RH_std    0.024716
3     t2m_std    0.024138
17    VPD_std    0.023356
20  sshf_mean    0.022749
7     u10_min    0.022514

Top 15 features selecionadas:
['d2m_mean', 'RH_min', 'sp_mean', 't2m_mean', 'RH_mean', 'd2m_std', 'v10_max', 't2m_max', 'ssrd_mean', 'e_mean', 'mes', 'u10_mean', 'v

In [16]:
#SE FOR OTIMIZAR TIRE O # DAS LINHAS DO CODIGO

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [4, 6, 8, 10],
    'min_samples_leaf': [3, 5, 10],
    'max_features': ['sqrt', 0.5]
}

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=10,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

print("\nTreinando GridSearch...")
#grid_search.fit(X_train_sel, y_train)

print("\nMelhores parâmetros:")
#print(grid_search.best_params_)

#best_model = grid_search.best_estimator_


#diagnosticar_overfitting(best_model, X_train_sel, y_train, X_test_sel, y_test)


Treinando GridSearch...

Melhores parâmetros:


In [17]:
model = RandomForestClassifier(max_depth= 3, max_features= 0.5, min_samples_leaf= 3, min_samples_split= 4, n_estimators= 100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=3, max_features=0.5, min_samples_leaf=3,
                       min_samples_split=4, random_state=42)

In [18]:
diagnosticar_overfitting(model, X_train, y_train, X_test, y_test)


================ DIAGNÓSTICO DE OVERFITTING ================
Acurácia Treino: 0.7849
Acurácia Teste:  0.7726
Diff Treino - Teste: 0.0123
-------------------------------------------------------------
Cross-Validation (média 10 folds): 0.7568
Desvio CV: 0.0638
-------------------------------------------------------------
Diagnóstico: OK — Sem sinais de overfitting.

